# late-search-budget -- Astra's candidate against eat-rest-v1, 32 paired seeds

Candidate: `external/candidates/eat-rest-v1-late-search-budget` (from `eat-rest-v1-late-search-budget.zip`). It is v1 plus one
block: a late-game gate (never before t=1300; needs a 20 s average of <=6 agents and <180 energy, plus either failed young
replacements after t=1400 or food-and-predator pressure after t=1600). Once active, a hungry young agent (energy < 80, age < 60)
that is searching with no fruit known and no predator within 180 moves slower, on a 20 s energy budget. Before the gate fires
the agent is v1, action for action. See `README.md` / `mechanism.md` in the candidate folder.

Both agents play seeds 11000-11031 through `external/candidates/endgame_probe.py`; the last block of cell 2 is the paired
per-seed difference. 32 seeds gives a standard error of about 45, so only a difference beyond roughly +-90 means anything.
Look at `late_activated` first: the gate cannot fire in a game that ends before t=1300-1400, and v1's mean on these seeds
is about 1200. Resumable; baseline games already stored in `logs/eg2/base` are reused. Nothing is written to `results/`.

**Cluster setup:** same as `parameter-tuning.ipynb` (`.env` with `GITHUB_TOKEN=<token>`).

In [1]:
import os

CLONE_DIR = "/home/jovyan/Nordic-AI-cup-2026"
if os.path.isdir(os.path.join(CLONE_DIR, ".git")):
    print(f"{CLONE_DIR} already cloned - skipping (use `git pull` there to update)")
else:
    # GitHub token is read from a git-ignored .env (GITHUB_TOKEN=...) in the kernel's cwd, or from the environment
    if os.path.isfile(".env"):
        for line in open(".env"):
            key, sep, value = line.strip().partition("=")
            if sep and not key.startswith("#"):
                os.environ.setdefault(key.strip(), value.strip().strip('"').strip("'"))
    TOKEN = os.environ.get("GITHUB_TOKEN")
    if not TOKEN:
        raise RuntimeError(f"GITHUB_TOKEN not set - create {os.path.abspath('.env')} containing GITHUB_TOKEN=<token>")
    !git clone https://{TOKEN}@github.com/sjoeen/Nordic-AI-cup-2026.git {CLONE_DIR}

/home/jovyan/Nordic-AI-cup-2026 already cloned - skipping (use `git pull` there to update)


In [2]:
import glob
import os
import subprocess
import sys

os.chdir(CLONE_DIR)
!git fetch origin challenge-1V2
!git checkout challenge-1V2
!git pull origin challenge-1V2

# Must run from survival-simulator/ so `src`, `agents`, `training` import.
if os.path.basename(os.getcwd()) != "survival-simulator":
    candidates = sorted({os.path.realpath(p) for p in glob.glob(os.path.join(os.getcwd(), "**", "survival-simulator"), recursive=True)
                         if os.path.isfile(os.path.join(p, "requirements.txt"))})
    if len(candidates) != 1:
        raise RuntimeError(f"cwd is {os.getcwd()}; found {len(candidates)} survival-simulator checkouts {candidates} - %cd into the right one")
    os.chdir(candidates[0])

print("cwd:", os.getcwd())
sys.path.insert(0, os.getcwd())
print(subprocess.run(["git", "log", "--oneline", "-1"], capture_output=True, text=True).stdout)

remote: Enumerating objects: 30, done.
remote: Counting objects: 100% (30/30), done.
remote: Compressing objects: 100% (14/14), done.
remote: Total 23 (delta 12), reused 20 (delta 9), pack-reused 0 (from 0)
Unpacking objects: 100% (23/23), 663.23 KiB | 12.51 MiB/s, done.
From https://github.com/sjoeen/Nordic-AI-cup-2026
 * branch            challenge-1V2 -> FETCH_HEAD
   fcb6821..ed9f20a  challenge-1V2 -> origin/challenge-1V2
M	Nordic-AI-Cup-2026-main/survival-simulator/results/index.csv
Already on 'challenge-1V2'
Your branch is behind 'origin/challenge-1V2' by 2 commits, and can be fast-forwarded.
  (use "git pull" to update your local branch)
From https://github.com/sjoeen/Nordic-AI-cup-2026
 * branch            challenge-1V2 -> FETCH_HEAD
Updating fcb6821..ed9f20a
Fast-forward
 .../survival-simulator/endgame-probe.ipynb         |  396 +-----
 .../candidates/eat-rest-endgame/survival_agent.py  |  156 ---
 .../eat-rest-v1-late-search-budget/README.md       |    9 +
 .../eat-rest-v1-la

In [3]:
!{sys.executable} -m pip install -r requirements.txt -r requirements-dev.txt

## 1. Baseline: eat-rest-v1 on 32 seeds (the 16 already stored are reused, ~4 min)

In [4]:
SEEDS = 32
!{sys.executable} -u external/candidates/endgame_probe.py --out logs/eg2/base --seeds {SEEDS} --brief 2>&1 | grep --line-buffered -v "pkg_resources\|pygame"

=== endgame probe: original-eat-rest-preserved, 16 games to run, 16 stored, overrides {} ===
  [  2.0 min] 8/16 games
  [  3.3 min] 16/16 games


## 2. Astra's candidate on the same seeds: full report, then the paired comparison (~8 min)

In [5]:
CANDIDATE = "eat-rest-v1-late-search-budget"
!{sys.executable} -u external/candidates/endgame_probe.py --out logs/eg2/{CANDIDATE} --candidate {CANDIDATE} --seeds {SEEDS} --compare logs/eg2/base 2>&1 | grep --line-buffered -v "pkg_resources\|pygame"

=== endgame probe: eat-rest-v1-late-search-budget, 32 games to run, 0 stored, overrides {} ===
  [  1.7 min] 8/32 games
  [  1.9 min] 16/32 games
  [  2.1 min] 24/32 games
  [  3.5 min] 32/32 games

32 games   extinction: mean 1318  sd 255  min 830  median 1292  max 1938   score mean 1310

-- world and colony by clock time (mean over the games still alive then)
     t games            n         pred        awake        ratio       sensed   threatened      endgame        trees       fruits fruit_energy       energy          age        speed
     0    32         5.00         0.00         0.00         0.00         0.00         0.00         0.00        26.97        32.28       651.62       150.03         0.10        10.00
   250    32         9.09         1.56         1.28         0.18         0.07         0.06         0.00        62.78       135.38      6858.22       221.15        66.75        11.43
   500    32         7.38         4.41         3.44         0.61         0.19         0.18

## Paired comparison only (no new games)

In [6]:
!{sys.executable} -u external/candidates/endgame_probe.py --out logs/eg2/{CANDIDATE} --candidate {CANDIDATE} --seeds {SEEDS} --compare logs/eg2/base --brief 2>&1 | grep --line-buffered -v "pkg_resources\|pygame"

=== endgame probe: eat-rest-v1-late-search-budget, 0 games to run, 32 stored, overrides {} ===

-- paired against logs/eg2/base on 32 shared seeds
  score: 1310 vs 1298   delta +11  SE 10  better on 10/32  worst -186  best +169
    end: 1318 vs 1305   delta +12  SE 10  better on 10/32  worst -185  best +172
  per seed (score delta): 11000:+0  11001:+56  11002:+142  11003:+0  11004:-186  11005:+18  11006:+0  11007:+169  11008:+0  11009:+0  11010:+0  11011:+0  11012:+0  11013:+0  11014:+0  11015:+0  11016:+0  11017:+0  11018:+0  11019:+10  11020:+0  11021:+0  11022:-15  11023:+0  11024:+116  11025:+15  11026:+24  11027:+0  11028:+17  11029:+0  11030:+0  11031:+0
  emergency_births                 mean      4.8   (in 30/32 games, mean there 5.1)
  late_activated                   mean      0.3   (in 32/32 games, mean there 0.3)
  late_activation_time             mean    506.1   (in 11/32 games, mean there 1472.4)
  late_requested_movement_saving   mean     88.1   (in 10/32 games, mean the